# Demo 01 : One agent and a growing tool list

Run the cells in order and watch. There is nothing to write.

Run the next cell first. It loads the course key and builds `llm`, the course model. Nothing below it depends on where the notebook runs.

In [1]:
# Course setup : run this cell first.
# On the classroom laptop the key comes from .env in the repository root.
# In Colab, add it under the key icon in the left sidebar as COURSE_API_KEY.
import os
import sys
import subprocess
import warnings
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
COURSE_FILES = "https://raw.githubusercontent.com/FeikoWielsma/Multi-Agents-LangGraph-Student/main/"
ROOT = None

if IN_COLAB:
    # Pinned to match uv.lock ; generated by build/stamp_setup.py
    PINS = ["langchain[mcp]==1.4.6", "langchain-core==1.6.12", "langchain-openai==1.7.1", "langgraph==1.2.17", "langgraph-checkpoint-sqlite==3.1.2", "deepagents[quickjs]==0.7.30", "langchain-quickjs==0.3.9", "langsmith==0.14.12", "openai==3.29.0", "python-dotenv==1.2.4", "langchain-huggingface==1.2.3", "sentence-transformers==6.1.0", "langchain-text-splitters==1.1.3", "rank-bm25==0.2.2", "agentevals==0.0.9", "httpx==0.28.1", "pytest==9.1.1", "pandas==3.0.6"]
    if not os.getenv("COURSE_SKIP_INSTALL"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *PINS], check=True)
    try:
        from google.colab import userdata
        os.environ.setdefault("COURSE_API_KEY", userdata.get("COURSE_API_KEY"))
    except Exception:
        pass
else:
    from dotenv import load_dotenv
    for folder in [Path.cwd(), *Path.cwd().parents]:
        if (folder / "pyproject.toml").exists():
            ROOT = folder
            load_dotenv(folder / ".env")
            break

PROVIDER = os.getenv("COURSE_PROVIDER", "course")      # "course" or "ollama"
if PROVIDER == "ollama":
    MODEL = os.getenv("COURSE_MODEL", "qwen3.5:4b")
    BASE_URL = "http://localhost:11434/v1"
    API_KEY = "ollama"
    THINKING_OFF = {"reasoning_effort": "none"}
else:
    MODEL = os.getenv("COURSE_MODEL", "openai/gpt-6-luna")  # xiaomi/mimo-v2.6-pro is the step up
    BASE_URL = os.getenv("COURSE_BASE_URL", "https://openrouter.ai/api/v1")
    API_KEY = os.getenv("COURSE_API_KEY")
    # OpenRouter has no Luna endpoint that honours temperature or parallel_tool_calls ;
    # seed is honoured, and routing prefers OpenAI so repeated runs hit one provider
    THINKING_OFF = {"reasoning": {"effort": "none"}, "provider": {"order": ["openai"]}}
    if not API_KEY:
        raise RuntimeError("COURSE_API_KEY is missing : copy .env.example to .env and paste the key")

from langchain_core._api import LangChainBetaWarning
warnings.filterwarnings("ignore", category=LangChainBetaWarning)

from langchain_openai import ChatOpenAI
from langchain_core.rate_limiters import InMemoryRateLimiter

# One shared limiter paces every model call in this process ; COURSE_RPS sets the budget.
# Paid OpenRouter models have no platform request cap (measured 26 October 2026)
RATE_LIMITER = InMemoryRateLimiter(requests_per_second=float(os.getenv("COURSE_RPS", "2")),
                                   check_every_n_seconds=0.1, max_bucket_size=1)


def make_llm(**kw):
    """The course chat model. Keyword arguments pass straight to ChatOpenAI."""
    kw.setdefault("model", MODEL)
    if PROVIDER == "ollama":
        kw.setdefault("temperature", 0)
    else:
        kw.setdefault("seed", 7)
    kw.setdefault("extra_body", THINKING_OFF)
    kw.setdefault("timeout", 90)        # seconds per request ; a stuck call raises instead of hanging
    kw.setdefault("rate_limiter", RATE_LIMITER)
    return ChatOpenAI(base_url=BASE_URL, api_key=API_KEY, **kw)


def make_embeddings(**kw):
    """Local sentence embeddings, so retrieval costs no API calls."""
    from langchain_huggingface import HuggingFaceEmbeddings
    kw.setdefault("model_name", "sentence-transformers/all-MiniLM-L6-v2")
    return HuggingFaceEmbeddings(**kw)


def data(name):
    """Path to a course data file ; downloaded when there is no local copy."""
    if ROOT and (ROOT / "data" / name).exists():
        return str(ROOT / "data" / name)
    target = Path(name)
    if not target.exists():
        import urllib.request
        target.parent.mkdir(parents=True, exist_ok=True)
        urllib.request.urlretrieve(COURSE_FILES + "data/" + name, target)
    return str(target)


def data_folder(name):
    """Path to a course data folder ; downloaded file by file, using its index.txt, when there is no local copy."""
    if ROOT and (ROOT / "data" / name).is_dir():
        return str(ROOT / "data" / name)
    for file in Path(data(f"{name}/index.txt")).read_text(encoding="utf-8").split():
        data(f"{name}/{file}")
    return name


llm = make_llm()
print(f"model {MODEL} via {PROVIDER}")

model openai/gpt-6-luna via course


We open the studio data of Comet Arcade : a database of games, assets, bugs, builds and playtests, and the design
docs. Comet Arcade is a made-up game studio. Its running project is Comet Snake.

In [2]:
import json
import re
import sqlite3
from pathlib import Path

from langchain.tools import tool

DB = data("game-studio/studio.db")
SECTIONS = []
for path in sorted(Path(data_folder("game-studio/docs")).glob("*.md")):
    title, *parts = path.read_text(encoding="utf-8").split("\n## ")
    SECTIONS += [f"{title.splitlines()[0]}\n## {part}" for part in parts]


def rows(sql, *args):
    with sqlite3.connect(DB) as conn:
        conn.row_factory = sqlite3.Row
        return [dict(row) for row in conn.execute(sql, args)]


def words(text):
    return set(re.findall(r"[a-z0-9-]{3,}", text.lower()))


def best(texts, query, k):
    q = words(query)
    return sorted(texts, key=lambda t: -len(q & words(t)))[:k]

print("sections:", len(SECTIONS), " games:", len(rows("SELECT * FROM games")), " bugs:", len(rows("SELECT * FROM bugs")))

sections: 30  games: 6  bugs: 9


We define nine tools for the studio assistant. Each is a small function over the data. The model reads only the
name, the arguments and the docstring.

In [3]:
@tool
def list_games(genre: str = "") -> str:
    """List Comet Arcade games with id, title, genre and status. Genre such as arcade or puzzle is optional."""
    return json.dumps(rows("SELECT game_id, title, genre, status FROM games WHERE genre LIKE ?", f"%{genre.strip().lower()}%"))

@tool
def game_info(title: str) -> str:
    """Look up a Comet Arcade game by its title or id. Returns id, title, genre, status and pitch."""
    t = title.strip()
    found = rows("SELECT game_id, title, genre, status, pitch FROM games WHERE game_id = ? OR title LIKE ?", t.upper(), f"%{t}%")
    return json.dumps(found[0]) if found else "Unknown game"

@tool
def assets_for_game(game_id: str) -> str:
    """List sprites, palettes and sounds for a game id such as CS-01, with colour and status."""
    return json.dumps(rows("SELECT asset_id, kind, name, colour, status FROM assets WHERE game_id = ?", game_id.strip().upper()))

@tool
def asset_info(asset_id: str) -> str:
    """Look up one asset by id such as CS-SPR-03 : kind, name, colour, size and status."""
    found = rows("SELECT * FROM assets WHERE asset_id = ?", asset_id.strip().upper())
    return json.dumps(found[0]) if found else "Unknown asset"

@tool
def bugs_for_game(game_id: str, status: str = "open") -> str:
    """List bugs for a game id such as CS-01. Status is open, fixed or all."""
    status = status.strip().lower()
    return json.dumps(rows("SELECT bug_id, title, severity, status, found_by FROM bugs WHERE game_id = ? AND (? = 'all' OR status = ?)",
                           game_id.strip().upper(), status, status))

@tool
def bug_details(bug_id: str) -> str:
    """Read one bug by id such as BUG-101, with severity, status and notes."""
    found = rows("SELECT * FROM bugs WHERE bug_id = ?", bug_id.strip().upper())
    return json.dumps(found[0]) if found else "Unknown bug"

@tool
def builds_for_game(game_id: str) -> str:
    """List the builds of a game id such as CS-01, with tests passed and status."""
    return json.dumps(rows("SELECT build, twist, tests_passed, tests_total, status, notes FROM builds WHERE game_id = ? ORDER BY build",
                           game_id.strip().upper()))

@tool
def playtests_for_game(game_id: str) -> str:
    """List playtests of a game id such as CS-01 : player, score, what ended the game and notes."""
    return json.dumps(rows("SELECT build, player, seed, score, ticks, ended_by, notes FROM playtests WHERE game_id = ?", game_id.strip().upper()))

@tool
def search_design_docs(query: str) -> str:
    """Search the design docs and the studio style guide. Returns the three best matching sections."""
    return "\n\n---\n\n".join(best(SECTIONS, query, 3))


tools = [list_games, game_info, assets_for_game, asset_info, bugs_for_game, bug_details,
         builds_for_game, playtests_for_game, search_design_docs]
by_name = {t.name: t for t in tools}

print("studio tools:", [t.name for t in tools])

studio tools: ['list_games', 'game_info', 'assets_for_game', 'asset_info', 'bugs_for_game', 'bug_details', 'builds_for_game', 'playtests_for_game', 'search_design_docs']


A real studio has many more systems : the storefront, the forums, analytics, marketing, the sound library. When one
agent serves the whole studio, their tools go into the same list. We add 51 such tools. None of them helps with
our questions, and each answers that it has no records. A few have tempting names, such as `player_reviews`.

In [4]:
from langchain_core.tools import StructuredTool
from pydantic import create_model

OTHER = [
    ("storefront_page", "Text and screenshots of a game's store page.", "game"),
    ("store_price", "Current price of a game in a region.", "region"),
    ("store_discounts", "Planned discounts for a sale event.", "event"),
    ("wishlist_count", "How many players wishlisted a game.", "game"),
    ("sales_by_region", "Copies sold in a region last month.", "region"),
    ("refund_rate", "Share of copies refunded for a game.", "game"),
    ("daily_players", "Players per day for a game.", "game"),
    ("session_length", "Average session length for a game.", "game"),
    ("retention_curve", "Share of players who return after a week.", "game"),
    ("crash_analytics", "Crash counts from released builds on player machines.", "game"),
    ("leaderboard_top", "Top scores on the public leaderboard.", "game"),
    ("achievement_list", "Achievements defined for a game.", "game"),
    ("achievement_rates", "Share of players with each achievement.", "game"),
    ("player_reviews", "Reviews that players left on the store page.", "game"),
    ("forum_threads", "Threads on the community forum, by topic.", "topic"),
    ("forum_moderation", "Posts waiting for a forum moderator.", "topic"),
    ("discord_mentions", "Mentions of a game on the community Discord.", "game"),
    ("social_posts", "Scheduled posts on the studio's social accounts.", "topic"),
    ("newsletter_drafts", "Drafts of the studio newsletter.", "month"),
    ("streamer_list", "Streamers who played a game.", "game"),
    ("press_kit", "Files in a game's press kit.", "game"),
    ("press_contacts", "Journalists who cover a genre.", "genre"),
    ("trailer_status", "Editing status of a game's trailer.", "game"),
    ("screenshot_library", "Marketing screenshots for a game.", "game"),
    ("event_calendar", "Game fairs the studio attends.", "year"),
    ("booth_plan", "Booth layout for a game fair.", "event"),
    ("localisation_status", "Translated strings per language.", "language"),
    ("translator_list", "Translators who work with the studio.", "language"),
    ("font_library", "Fonts licensed for use in games.", "style"),
    ("music_tracks", "Tracks in the studio's music library.", "mood"),
    ("sound_library", "Licensed sound effects by keyword.", "keyword"),
    ("voice_actors", "Voice actors and the characters they play.", "game"),
    ("controller_support", "Controllers tested with a game.", "game"),
    ("platform_rules", "Certification rules of a console platform.", "platform"),
    ("platform_submissions", "Status of a submission to a platform.", "platform"),
    ("age_ratings", "Age rating of a game per region.", "region"),
    ("accessibility_options", "Accessibility settings a game offers.", "game"),
    ("save_file_format", "Layout of a game's save file.", "game"),
    ("cloud_save_usage", "Cloud storage used by a game's saves.", "game"),
    ("server_status", "Status of the studio's game servers.", "region"),
    ("patch_notes", "Published patch notes for a game.", "game"),
    ("release_calendar", "Planned release dates.", "quarter"),
    ("team_directory", "Name and role of a member of the crew.", "name"),
    ("meeting_notes", "Notes of the weekly studio meeting.", "date"),
    ("office_calendar", "Who is in the studio on a date.", "date"),
    ("hardware_inventory", "Test phones and consoles in the studio.", "device"),
    ("license_keys", "Software licences held by the studio.", "product"),
    ("backup_status", "Last backup of a project folder.", "project"),
    ("build_server_queue", "Jobs waiting on the build server.", "project"),
    ("competitor_games", "Games similar to a given pitch.", "pitch"),
    ("genre_trends", "Top selling genres this month.", "month"),
]


def other_system(name, description, arg):
    return StructuredTool.from_function(
        func=lambda **kw: f"no records for {kw[arg]}", name=name, description=description,
        args_schema=create_model(name, **{arg: (str, ...)}))


OTHER_TOOLS = [other_system(*spec) for spec in OTHER]
print("other tools:", len(OTHER_TOOLS))

other tools: 51


We use six questions over the studio's data, each with the facts a correct answer contains. `Meter` counts model
calls and tokens through a callback, and `correct` checks the facts. A `|` separates accepted alternatives. The
last question needs two steps : the playtests name the player, and the assets name the sprite.

In [5]:
from langchain_core.callbacks import BaseCallbackHandler


class Meter(BaseCallbackHandler):
    def __init__(self):
        self.calls = self.input = self.output = 0

    def on_llm_end(self, response, **kwargs):
        usage = response.generations[0][0].message.usage_metadata or {}
        self.calls += 1
        self.input += usage.get("input_tokens", 0)
        self.output += usage.get("output_tokens", 0)

    def __str__(self):
        return f"{self.calls} calls, {self.input:,} input tokens, {self.output:,} output tokens"


def correct(answer, expected):
    text = answer.lower()
    return all(any(alt in text for alt in item.split("|")) for item in expected)

QUESTIONS = [
    ("Which Comet Arcade games are puzzle games?", ["star slide", "orbit blocks", "gravity golf"]),
    ("What colour is sprite CS-SPR-03, and is it final?", ["#3a4a7a", "draft|not final"]),
    ("Which bugs are open on Comet Snake?", ["bug-101", "bug-104", "bug-105"]),
    ("How many tests did build 2 of Comet Snake pass, and why did it fail?", ["18", "self hit|last body cell"]),
    ("What is the best bot score in the Comet Snake playtests, and what ended that game?", ["58", "self|own body|itself"]),
    ("Which human playtester could not see the comet that hit them, and which sprite is to blame?", ["mira", "cs-spr-03"]),
]

## a · one-agent-limits

We build one agent with `n` tools : the nine studio tools plus the first `n - 9` others, shuffled so the studio
tools are not all at the front. It asks every question once and prints a line per question. A detour is a call to
one of the other systems' tools.

In [6]:
import random

from langchain.agents import create_agent

PROMPT = "You are the studio assistant of Comet Arcade. Answer from the tools only. Be brief."
STUDIO_NAMES = {t.name for t in tools}
results = []


def measure(n):
    chosen = tools + OTHER_TOOLS[:n - len(tools)]
    random.Random(n).shuffle(chosen)
    agent = create_agent(llm, tools=chosen, system_prompt=PROMPT)
    row = {"tools": n, "correct": 0, "calls": 0, "input": 0, "output": 0, "detours": 0}
    for question, expected in QUESTIONS:
        meter = Meter()
        messages = agent.invoke({"messages": [{"role": "user", "content": question}]},
                                {"callbacks": [meter]})["messages"]
        used = [m.name for m in messages if m.type == "tool"]
        ok = correct(messages[-1].content, expected)
        detours = sum(name not in STUDIO_NAMES for name in used)
        for key, value in [("correct", ok), ("calls", meter.calls), ("input", meter.input),
                           ("output", meter.output), ("detours", detours)]:
            row[key] += value
        print(f"{'ok' if ok else 'WRONG':5} {meter.calls} calls {meter.input:>7,} in  {used}")
        print("answer:", messages[-1].content)
    results.append(row)
    print(f"{n} tools : {row['correct']}/{len(QUESTIONS)} correct, {row['calls']} calls, "
          f"{row['input']:,} input tokens, {row['detours']} detours")

We start with the nine studio tools alone. In this baseline every tool is relevant and the tool list is short.

In [7]:
measure(9)

ok    2 calls     916 in  ['list_games']
answer: - Star Slide (released)
- Orbit Blocks (in development)
- Gravity Golf (concept)


ok    2 calls     898 in  ['asset_info']
answer: CS-SPR-03 is **#3a4a7a** and is **not final**; its status is draft.


ok    3 calls   1,521 in  ['game_info', 'bugs_for_game']
answer: Open bugs on **Comet Snake**:

- **BUG-101** — Comet hard to see on the dark background *(blocker)*
- **BUG-104** — Shield pickup does nothing *(major)*
- **BUG-105** — Comet hit sound plays twice *(minor)*


ok    3 calls   1,565 in  ['game_info', 'builds_for_game']
answer: Build 2 passed **18 of 19 tests**. It failed because a self-hit at the last body cell was missed.


ok    3 calls   1,797 in  ['game_info', 'playtests_for_game']
answer: The best bot score was **58**; the game ended by **self-collision**.


ok    3 calls   2,337 in  ['playtests_for_game', 'assets_for_game']
answer: Mira couldn’t see the comet that hit her. The likely culprit is the comet sprite, **CS-SPR-03** (currently draft).
9 tools : 6/6 correct, 16 calls, 9,034 input tokens, 0 detours


With twenty tools, eleven belong to other systems.

In [8]:
measure(20)

ok    2 calls   1,406 in  ['list_games']
answer: - Star Slide — released
- Orbit Blocks — in development
- Gravity Golf — concept


ok    2 calls   1,388 in  ['asset_info']
answer: CS-SPR-03 is **#3a4a7a** and is **not final**; its status is draft.


ok    3 calls   2,256 in  ['game_info', 'bugs_for_game']
answer: Open bugs on Comet Snake:

- **BUG-101 — Blocker:** Comet is hard to see on the dark background.
- **BUG-104 — Major:** Shield pickup does nothing.
- **BUG-105 — Minor:** Comet hit sound plays twice.


ok    3 calls   2,300 in  ['game_info', 'builds_for_game']
answer: Build 2 passed 18 of 19 tests. It failed because a self-hit wasn’t detected at the last body cell.


ok    2 calls   1,712 in  ['playtests_for_game']
answer: The best bot score was **58**, and the game ended by **self-collision**.


ok    3 calls   3,072 in  ['playtests_for_game', 'assets_for_game']
answer: Mira couldn’t see the comet that hit her. The likely culprit is the comet sprite, **CS-SPR-03**, which is dark blue (#3a4a7a) against the deep-space palette (#0b1030).
20 tools : 6/6 correct, 15 calls, 12,134 input tokens, 0 detours


We go up to forty tools.

In [9]:
measure(40)

ok    2 calls   2,268 in  ['list_games']
answer: - **Star Slide** — released
- **Orbit Blocks** — in development
- **Gravity Golf** — concept


ok    2 calls   2,250 in  ['asset_info']
answer: Sprite CS-SPR-03 is **#3a4a7a** and is still a **draft**, not final.


ok    3 calls   3,549 in  ['game_info', 'bugs_for_game']
answer: Open bugs on **Comet Snake**:

- **BUG-101** — Comet hard to see on the dark background *(blocker)*
- **BUG-104** — Shield pickup does nothing *(major)*
- **BUG-105** — Comet hit sound plays twice *(minor)*


ok    2 calls   2,343 in  ['builds_for_game']
answer: Build 2 passed 18 of 19 tests. It failed because a self-hit at the last body cell was missed.


ok    2 calls   2,574 in  ['playtests_for_game']
answer: The best bot score was **58**, and that game ended by **self-collision**.


ok    3 calls   4,365 in  ['playtests_for_game', 'assets_for_game']
answer: Mira was the human playtester who couldn’t see the comet. The likely culprit is the **comet sprite, CS-SPR-03**—it’s a dark blue and still in draft.
40 tools : 6/6 correct, 14 calls, 17,349 input tokens, 0 detours


We reach sixty tools : nine studio tools and 51 others.

In [10]:
measure(60)

ok    2 calls   3,132 in  ['list_games']
answer: - Star Slide — released
- Orbit Blocks — in development
- Gravity Golf — concept


ok    2 calls   3,114 in  ['asset_info']
answer: CS-SPR-03 is **#3a4a7a**. It’s **still a draft**, not final.


ok    3 calls   4,845 in  ['game_info', 'bugs_for_game']
answer: Open bugs on **Comet Snake**:

- **BUG-101** — Comet hard to see on the dark background *(blocker)*
- **BUG-104** — Shield pickup does nothing *(major)*
- **BUG-105** — Comet hit sound plays twice *(minor)*


ok    3 calls   4,889 in  ['game_info', 'builds_for_game']
answer: Build 2 passed **18 of 19 tests**. It failed because **a self-hit at the last body cell was missed**.


ok    3 calls   5,121 in  ['game_info', 'playtests_for_game']
answer: The best bot score was **58**, and the game ended by **self-collision**.


ok    3 calls   5,661 in  ['playtests_for_game', 'assets_for_game']
answer: Mira couldn’t see the comet that hit her. The likely culprit is the draft comet sprite **CS-SPR-03** (colour **#3a4a7a**), which may blend into the deep-space palette.
60 tools : 6/6 correct, 16 calls, 26,762 input tokens, 0 detours


We compare the four runs, including average input tokens per call. The model receives the full tool list on every
call, including unused tools.

In [11]:
print(f"{'tools':>5} {'correct':>8} {'calls':>6} {'input':>8} {'per call':>9} {'output':>7} {'detours':>8}")
for r in results:
    print(f"{r['tools']:>5} {r['correct']:>6}/{len(QUESTIONS)} {r['calls']:>6} {r['input']:>8,} "
          f"{r['input'] // r['calls']:>9,} {r['output']:>7} {r['detours']:>8}")

tools  correct  calls    input  per call  output  detours
    9      6/6     16    9,034       564     426        0
   20      6/6     15   12,134       808     418        0
   40      6/6     14   17,349     1,239     400        0
   60      6/6     16   26,762     1,672     443        0


We compare correctness, calls and tokens before splitting the agent. More tool descriptions add input context, but
correctness need not decrease. One run per question is a small sample, and the fact check only matches text. We
also read each answer for contradictions and unsupported claims.

## Things to try

Optional.

- Rename three of the other tools so they overlap with the studio ones, for example `issue_tracker` or
  `art_library`, and measure again
- Run `measure(60)` with `make_llm(model="xiaomi/mimo-v2.6-pro")` in place of `llm`
- Shorten the docstring of `builds_for_game` to two words and see which questions fail